# Optimization Modes Comparison

This notebook provides a structured comparison of the optimization modes used in the SSF workflow. It looks at both the a-side and b-side formulations, and evaluates how different optimizer and backend combinations behave across several $(n, k)$ settings and p-rules.

The goal is not only to identify the best-performing configuration in each case, but also to show how the same problem can be approached with different search strategies. Some combinations rely on exhaustive search, while others use beam-based optimization with either dense or procedural Monte Carlo backends.

The results are summarized as winning scores for each setup, so a reader can quickly compare which optimization mode performs best for a given problem size and parameter choice. This makes the notebook useful both as a demonstration of the available modes and as a reference for selecting a practical optimization approach in future runs.




https://github.com/robhendrik/SSF


## Table of Contents

- [1. Overview](#sec-overview)
- [2. Exhaustive optimizer + analytical backend](#sec-exh-analytical)
- [3. Exhaustive optimizer + dense exhaustive backend](#sec-exh-dense)
- [4. Beam optimizer + dense MC backend](#sec-beam-dense)
- [5. Beam optimizer + procedural MC backend](#sec-beam-procedural)
- [6. Build comparison table](#sec-build-table)


<a id="sec-overview"></a>

# 1. Overview


In [1]:
import json
import subprocess
import sys
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
output_dir = repo_root / "work" / "benchmarks" / "optimizer_benchmark_matrix"
json_dir = repo_root / "work" / "benchmarks" / "ark_matrix"
output_dir.mkdir(parents=True, exist_ok=True)
json_dir.mkdir(parents=True, exist_ok=True)

exhaustive_script = repo_root / "scripts" / "optimize" / "run_exhaustive_strategy_optimizer.py"
beam_script = repo_root / "scripts" / "optimize" / "run_beam_strategy_optimizer.py"

families = ["majority", "pyramid", "horizontal", "vertical"]
p_rules = [0.7, 0.8, 0.9]

In [2]:
def run_checked(cmd: list[str]) -> None:
    completed = subprocess.run(
        cmd,
        cwd=repo_root,
        capture_output=True,
        text=True,
    )

    if completed.returncode != 0:
        message = [
            "Subprocess failed.",
            f"Command: {' '.join(cmd)}",
            f"Return code: {completed.returncode}",
            "\nSTDOUT:\n" + completed.stdout,
            "\nSTDERR:\n" + completed.stderr,
        ]
        raise RuntimeError("\n".join(message))

runs: list[dict[str, object]] = []

<a id="sec-exh-analytical"></a>

# 2. Exhaustive optimizer + analytical backend

In [3]:
for n2, k_box in [(4, 1), (4, 3), (8, 1), (8, 7), (16, 1), (16, 15)]:
    json_file = json_dir / f"exhaustive_analytical_n2_{n2}_k_{k_box}.json"
    cmd = [
        sys.executable,
        str(exhaustive_script),
        "--mode",
        "analytical",
        "--p-rules",
        *[str(v) for v in p_rules],
        "--n2-values",
        str(n2),
        "--k-box-values",
        str(k_box),
        "--families",
        *families,
        "--top-k",
        "10",
        "--cache-enabled",
        "--json-output",
        str(json_file),
        "--quiet",
    ]

    run_checked(cmd)
    runs.append(
        {
            "optimizer": "exhaustive",
            "backend": "analytical",
            "n2": n2,
            "k_box": k_box,
            "json_file": json_file,
        }
    )


<a id="sec-exh-dense"></a>

# 3. Exhaustive optimizer + dense exhaustive backend

In [4]:
for n2, k_box in [(4, 1), (4, 3)]:
    json_file = json_dir / f"exhaustive_dense_exhaustive_n2_{n2}_k_{k_box}.json"
    cmd = [
        sys.executable,
        str(exhaustive_script),
        "--mode",
        "dense_exhaustive",
        "--p-rules",
        *[str(v) for v in p_rules],
        "--n2-values",
        str(n2),
        "--k-box-values",
        str(k_box),
        "--families",
        *families,
        "--top-k",
        "10",
        "--cache-enabled",
        "--json-output",
        str(json_file),
        "--quiet",
    ]

    run_checked(cmd)
    runs.append(
        {
            "optimizer": "exhaustive",
            "backend": "dense_exhaustive",
            "n2": n2,
            "k_box": k_box,
            "json_file": json_file,
        }

    )



<a id="sec-beam-dense"></a>

# 4. Beam optimizer + dense MC backend

In [5]:
for n2, k_box in [(4, 1), (4, 3), (8, 1), (8, 7)]:
    for p_rule in p_rules:
        json_file = json_dir / f"beam_dense_mc_n2_{n2}_k_{k_box}_p_{str(p_rule).replace('.', '_')}.json"
        cmd = [
            sys.executable,
            str(beam_script),
            "--mode",
            "dense_mc",
            "--p-rule",
            str(p_rule),
            "--n2-values",
            str(n2),
            "--k-box-values",
            str(k_box),
            "--families",
            *families,
            "--beam-width",
            "8",
            "--generations",
            "4",
            "--mutations-per-candidate",
            "4",
            "--top-k",
            "10",
            "--seed",
            "123",
            "--cache-enabled",
            "--json-output",
            str(json_file),
            "--quiet",
        ]

        run_checked(cmd)
        runs.append(
            {
                "optimizer": "beam",
                "backend": "dense_mc",
                "n2": n2,
                "k_box": k_box,
                "p_rule": float(p_rule),
                "json_file": json_file,
            }
        )




<a id="sec-beam-procedural"></a>

# 5. Beam optimizer + procedural MC backend

In [6]:
for n2, k_box in [(4, 1), (4, 3), (8, 1), (8, 7), (16, 1)]:
    for p_rule in p_rules:
        json_file = json_dir / f"beam_procedural_mc_n2_{n2}_k_{k_box}_p_{str(p_rule).replace('.', '_')}.json"
        cmd = [
            sys.executable,
            str(beam_script),
            "--mode",
            "procedural_mc",
            "--p-rule",
            str(p_rule),
            "--n2-values",
            str(n2),
            "--k-box-values",
            str(k_box),
            "--families",
            *families,
            "--beam-width",
            "8",
            "--generations",
            "4",
            "--mutations-per-candidate",
            "4",
            "--top-k",
            "10",
            "--seed",
            "123",
            "--cache-enabled",
            "--json-output",
            str(json_file),
            "--quiet",
        ]

        run_checked(cmd)

        runs.append(
            {
                "optimizer": "beam",
                "backend": "procedural_mc",
                "n2": n2,
                "k_box": k_box,
                "p_rule": float(p_rule),
                "json_file": json_file,
            }
        )


<a id="sec-build-table"></a>

# 6. Build comparison table

In [7]:
rows: list[dict[str, object]] = []

for run in runs:
    payload = json.loads(Path(run["json_file"]).read_text(encoding="utf-8"))

    if run["optimizer"] == "exhaustive":
        for result_block in payload.get("results", []):
            p_rule = float(result_block["p_rule"])
            for item in result_block.get("ranked_results", []):
                rows.append(
                    {
                        "optimizer": "exhaustive",
                        "backend": str(run["backend"]),
                        "n2": int(run["n2"]),
                        "k_box": int(run["k_box"]),
                        "p_rule": p_rule,
                        "rank": int(item["rank"]),
                        "success": float(item["success"]),
                        "candidate_name": str(item["candidate_name"]),
                        "json_file": str(run["json_file"]),
                    }
                )
    else:
        p_rule = float(run["p_rule"])
        for item in payload.get("final_results", []):
            rows.append(
                {
                    "optimizer": "beam",
                    "backend": str(run["backend"]),
                    "n2": int(run["n2"]),
                    "k_box": int(run["k_box"]),
                    "p_rule": p_rule,
                    "rank": int(item["rank"]),
                    "success": float(item["success"]),
                    "candidate_name": str(item["candidate_name"]),
                    "json_file": str(run["json_file"]),
                }
            )

df = pd.DataFrame(
    rows,
    columns=[
        "optimizer",
        "backend",
        "n2",
        "k_box",
        "p_rule",
        "rank",
        "success",
        "candidate_name",
        "json_file",
    ],
)

df = df.sort_values(["backend", "n2", "k_box", "p_rule", "rank"], kind="mergesort").reset_index(drop=True)


In [8]:
from pathlib import Path

results_dir = repo_root / "results" / "optimizer_benchmark_matrix"
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "optimizer_benchmark_matrix.csv"
parquet_path = results_dir / "optimizer_benchmark_matrix.parquet"
pkl_path = results_dir / "optimizer_benchmark_matrix.pkl"

df.to_csv(csv_path, index=False)
print(f"Saved CSV: {csv_path.relative_to(repo_root)}")

df.to_pickle(pkl_path)
print(f"Saved PKL: {pkl_path.relative_to(repo_root)}")

print(f"Rows saved: {len(df)}")

Saved CSV: results\optimizer_benchmark_matrix\optimizer_benchmark_matrix.csv
Saved PKL: results\optimizer_benchmark_matrix\optimizer_benchmark_matrix.pkl
Rows saved: 363


In [9]:
import pandas as pd
from pathlib import Path

# Load df from the saved CSV if it is not already in memory.
if "df" not in globals() or not isinstance(df, pd.DataFrame):
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    csv_path = repo_root / "results" / "optimizer_benchmark_matrix" / "optimizer_benchmark_matrix.csv"
    df = pd.read_csv(csv_path)

# Winning score only: keep rank=1 rows and compare methods on success.
winners = df[df["rank"] == 1].copy()
winners["method"] = winners["optimizer"].map({"exhaustive": "exhaustive", "beam": "beam"}) + " " + winners["backend"].map({
    "dense_exhaustive": "dense",
    "dense_mc": "dense",
    "procedural_mc": "procedural",
    "analytical": "analytical",
})

pivot = winners.pivot_table(
    index=["n2", "k_box", "p_rule"],
    columns="method",
    values="success",
    aggfunc="max",
)

column_order = ["exhaustive dense", "beam dense", "beam procedural", "exhaustive analytical"]
pivot = pivot.reindex(columns=[col for col in column_order if col in pivot.columns])
pivot.columns.name = None
pivot.index.names = ["n2", "k_box", "p_rule"]

pivot.round(2)


exhaustive dense  beam dense  beam procedural  \
n2 k_box p_rule                                                  
4  1     0.7                 0.69        0.69             0.69   
         0.8                 0.70        0.70             0.70   
         0.9                 0.73        0.73             0.73   
   3     0.7                 0.69        0.69             0.69   
         0.8                 0.70        0.70             0.70   
         0.9                 0.82        0.82             0.82   
8  1     0.7                  NaN        0.63             0.64   
         0.8                  NaN        0.64             0.64   
         0.9                  NaN        0.65             0.65   
   7     0.7                  NaN        0.64             0.64   
         0.8                  NaN        0.64             0.64   
         0.9                  NaN        0.76             0.76   
16 1     0.7                  NaN         NaN             0.60   
         0.8                  NaN         NaN             0.60   
         0.9                  NaN         NaN             0.61   
   15    0.7                  NaN         NaN              NaN   
         0.8                  NaN         NaN              NaN   
         0.9                  NaN         NaN              NaN   

                 exhaustive analytical  
n2 k_box p_rule                         
4  1     0.7                      0.69  
         0.8                      0.70  
         0.9                      0.72  
   3     0.7                      0.69  
         0.8                      0.70  
         0.9                      0.82  
8  1     0.7                      0.64  
         0.8                      0.64  
         0.9                      0.65  
   7     0.7                      0.64  
         0.8                      0.64  
         0.9                      0.76  
16 1     0.7                      0.60  
         0.8                      0.60  
         0.9                      0.61  
   15    0.7                      0.60  
         0.8                      0.60  
         0.9                      0.70

## Reference

- Author: Rob Hendriks
- Copyright: © 2026 SSF contributors
- GitHub: https://github.com/robhendrik/SSF